# Phase 5 — Python Data Processing

## Meridian Retail Group Customer Churn Capstone

This phase uses Python, Pandas and NumPy to process and analyze the
Meridian Retail Group datasets.

The objectives are to:

- Load and inspect the five datasets.
- Demonstrate reusable Python functions.
- Use loops and comprehensions appropriately.
- Apply conditional logic.
- Demonstrate exception handling.
- Safely parse dates and numerical values.
- Use lambda functions where appropriate.
- Apply NumPy vectorized numerical operations.
- Create customer-level analytical features.
- Save processed datasets for subsequent analysis.



In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
from google.colab import drive
drive.mount ('/content/drive')

Mounted at /content/drive


In [3]:
# Locating project folder
MY_DRIVE = '/content/drive/MyDrive'

items = os.listdir(MY_DRIVE)

meridian_folders = [
    item for item in items
    if item.startswith('Meridian Retail Group')
]

if not meridian_folders:
    raise FileNotFoundError(
        "Meridian Retail Group project folder was not found."
    )

meridian_folder = meridian_folders[0]

print("Project folder found:")
print(meridian_folder)


Project folder found:
Meridian Retail Group - Customer Churn Capstone 


In [4]:
# BUILDING a reusable file path

meridian_path = os.path.join(
    MY_DRIVE,
    meridian_folder
)

data_path = os.path.join(
    meridian_path,
    'Data'
)

raw_data_path = os.path.join(
    data_path,
    'Raw Data'
)

processed_data_path = os.path.join(
    data_path,
    'Processed Data'
)

print("Project path:", meridian_path)
print("Raw data path:", raw_data_path)
print("Processed data path:", processed_data_path)

Project path: /content/drive/MyDrive/Meridian Retail Group - Customer Churn Capstone 
Raw data path: /content/drive/MyDrive/Meridian Retail Group - Customer Churn Capstone /Data/Raw Data
Processed data path: /content/drive/MyDrive/Meridian Retail Group - Customer Churn Capstone /Data/Processed Data


In [5]:
os.makedirs(processed_data_path, exist_ok=True)
print("Processed data path is ready:")
print(processed_data_path)

Processed data path is ready:
/content/drive/MyDrive/Meridian Retail Group - Customer Churn Capstone /Data/Processed Data


In [6]:
# LOADING the five datasets

file_names = {
    'customers': 'customers.csv',
    'products': 'products.csv',
    'orders': 'orders.csv',
    'order_items': 'order_items.csv',
    'support_tickets': 'support_tickets.csv'
}

In [7]:
# CREATING a reusable loading function

def load_csv_dataset(dataset_name, file_name, folder_path):
    """
    Load a CSV dataset from the specified folder.
    """

    file_path = os.path.join(
        folder_path,
        file_name
    )

    try:
        df = pd.read_csv(file_path)
        print(f"{dataset_name} loaded successfully.")
        return df

    except FileNotFoundError:
        print(f"File not found: {file_name}")
        return None

    except Exception as error:
        print(f"Error loading {file_name}: {error}")
        return None

In [8]:
# USING a LOOP to LOAD everything
datasets = {}

for dataset_name, file_name in file_names.items():
    datasets[dataset_name] = load_csv_dataset(
        dataset_name,
        file_name,
        raw_data_path
    )

customers loaded successfully.
products loaded successfully.
orders loaded successfully.
order_items loaded successfully.
support_tickets loaded successfully.


In [9]:
# CHECKING if everything loaded
for name, df in datasets.items():
    if df is not None:
        print(
            f"{name}: "
            f"{df.shape[0]} rows × {df.shape[1]} columns"
        )

customers: 6090 rows × 14 columns
products: 220 rows × 9 columns
orders: 28297 rows × 9 columns
order_items: 73505 rows × 5 columns
support_tickets: 5200 rows × 8 columns


In [10]:
# CREATING a convinent datasets variables

customers = datasets['customers']
products = datasets['products']
orders = datasets['orders']
order_items = datasets['order_items']
support_tickets = datasets['support_tickets']

In [11]:
# CREATING a RESUSABLE INSPECTION FUNCTION
def inspect_dataset(name, df):
    """
    Display basic information about a dataset.
    """

    print("=" * 70)
    print(name.upper())
    print("=" * 70)

    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    print("\nColumn Names:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print()

In [12]:
# USING the FUNCTION with a LOOP

for name, df in datasets.items():
    inspect_dataset(name, df)

CUSTOMERS
Rows: 6090
Columns: 14

Column Names:
['customer_id', 'first_name', 'last_name', 'email', 'phone', 'gender', 'date_of_birth', 'signup_date', 'state', 'city', 'membership_tier', 'preferred_channel', 'marketing_opt_in', 'referral_source']

Data Types:
customer_id          object
first_name           object
last_name            object
email                object
phone                object
gender               object
date_of_birth        object
signup_date          object
state                object
city                 object
membership_tier      object
preferred_channel    object
marketing_opt_in     object
referral_source      object
dtype: object

Missing Values:
customer_id             0
first_name              0
last_name               0
email                 242
phone                 423
gender                  0
date_of_birth         181
signup_date             0
state                   0
city                  121
membership_tier         0
preferred_channel       0
marke

In [13]:
# Dataset name COMPREHENSION
dataset_names = [
    name.upper()
    for name in datasets.keys()
]

print("Datasets being processed:")
print(dataset_names)

Datasets being processed:
['CUSTOMERS', 'PRODUCTS', 'ORDERS', 'ORDER_ITEMS', 'SUPPORT_TICKETS']


In [14]:
# SAFE DATE Processing
def safe_date_conversion(df, column):
    """
    Safely convert a column to datetime.
    Invalid dates are converted to NaT.
    """

    try:
        df[column] = pd.to_datetime(
            df[column],
            errors='coerce'
        )

        invalid_count = df[column].isna().sum()

        print(
            f"{column}: conversion completed. "
            f"Invalid/missing dates: {invalid_count}"
        )

    except KeyError:
        print(f"Column '{column}' does not exist.")

    except Exception as error:
        print(
            f"Error converting '{column}': {error}"
        )

    return df

In [15]:
# Processing all data columns

date_columns = {
    'customers': [
        'date_of_birth',
        'signup_date'
    ],

    'products': [
        'launch_date'
    ],

    'orders': [
        'order_date'
    ],

    'support_tickets': [
        'ticket_date'
    ]
}

In [16]:
# Using nested loops
for dataset_name, columns in date_columns.items():

    for column in columns:

        datasets[dataset_name] = safe_date_conversion(
            datasets[dataset_name],
            column
        )

date_of_birth: conversion completed. Invalid/missing dates: 181
signup_date: conversion completed. Invalid/missing dates: 487
launch_date: conversion completed. Invalid/missing dates: 0
order_date: conversion completed. Invalid/missing dates: 424
ticket_date: conversion completed. Invalid/missing dates: 0


In [17]:
# Refreshing variables
customers = datasets['customers']
products = datasets['products']
orders = datasets['orders']
order_items = datasets['order_items']
support_tickets = datasets['support_tickets']

In [18]:
# SAFE NUMERICAL Processing
def safe_numeric_conversion(df, columns):
    """
    Safely convert selected columns to numeric values.
    Invalid values are converted to NaN.
    """

    for column in columns:

        try:
            df[column] = pd.to_numeric(
                df[column],
                errors='coerce'
            )

            invalid_count = df[column].isna().sum()

            print(
                f"{column}: conversion completed. "
                f"Missing/invalid values: {invalid_count}"
            )

        except KeyError:
            print(
                f"Column '{column}' does not exist."
            )

        except Exception as error:
            print(
                f"Error converting '{column}': {error}"
            )

    return df

In [19]:
print("Product columns")
print(products.columns.tolist)

Product columns
<bound method IndexOpsMixin.tolist of Index(['product_id', 'product_name', 'category', 'sub_category', 'brand',
       'unit_cost', 'unit_price', 'launch_date', 'is_active'],
      dtype='object')>


In [20]:
print("Order columns")
print(orders.columns.tolist)

Order columns
<bound method IndexOpsMixin.tolist of Index(['order_id', 'customer_id', 'order_date', 'channel', 'order_status',
       'payment_method', 'shipping_cost', 'discount_pct', 'total_amount'],
      dtype='object')>


In [21]:
print("Order items columns")
print(order_items.columns.tolist)

Order items columns
<bound method IndexOpsMixin.tolist of Index(['order_item_id', 'order_id', 'product_id', 'quantity',
       'unit_price_at_purchase'],
      dtype='object')>


In [22]:
print("Support Tickets columns")
print(support_tickets.columns.tolist)

Support Tickets columns
<bound method IndexOpsMixin.tolist of Index(['ticket_id', 'customer_id', 'ticket_date', 'issue_category', 'channel',
       'resolution_time_hours', 'satisfaction_score', 'ticket_text'],
      dtype='object')>


In [23]:
# DEFINING numerical columns
numeric_columns = {
    'products': [
        'unit_cost',
        'unit_price'
    ],

    'orders': [
        'shipping_cost',
        'discount_pct',
        'total_amount'
    ],

    'order_items': [
        'quantity',
        'unit_price_at_purchase'
    ],

    'support_tickets': [
        'resolution_time_hours',
        'satisfaction_score'
    ]
}

In [24]:
# Process numerical columns

for dataset_name, columns in numeric_columns.items():

    datasets[dataset_name] = safe_numeric_conversion(
        datasets[dataset_name],
        columns
    )

unit_cost: conversion completed. Missing/invalid values: 0
unit_price: conversion completed. Missing/invalid values: 22
shipping_cost: conversion completed. Missing/invalid values: 0
discount_pct: conversion completed. Missing/invalid values: 0
total_amount: conversion completed. Missing/invalid values: 0
quantity: conversion completed. Missing/invalid values: 0
unit_price_at_purchase: conversion completed. Missing/invalid values: 1469
resolution_time_hours: conversion completed. Missing/invalid values: 0
satisfaction_score: conversion completed. Missing/invalid values: 592


In [25]:
# Refreshing variables again
customers = datasets['customers']
products = datasets['products']
orders = datasets['orders']
order_items = datasets['order_items']
support_tickets = datasets['support_tickets']

In [26]:
print("Customer columns")
print(customers.columns.tolist)

Customer columns
<bound method IndexOpsMixin.tolist of Index(['customer_id', 'first_name', 'last_name', 'email', 'phone', 'gender',
       'date_of_birth', 'signup_date', 'state', 'city', 'membership_tier',
       'preferred_channel', 'marketing_opt_in', 'referral_source'],
      dtype='object')>


In [27]:
# NUMPY Vectorized Customer Age
today = pd.Timestamp.today()

customers['Age'] = (
    today.year
    - customers['date_of_birth'].dt.year
    - (
        (today.month < customers['date_of_birth'].dt.month)
        |
        (
            (today.month == customers['date_of_birth'].dt.month)
            &
            (today.day < customers['date_of_birth'].dt.day)
        )
    )
)

print("Customer age calculated successfully.")

# this is vectorized because pandas performs
# the calculation across the entire column

Customer age calculated successfully.


In [28]:
# Using numpy to validate age
customers["Age"] = np.where(
    customers["Age"].between(18, 100),
    customers["Age"],
    np.nan
)

print("Customer age validation completed")

Customer age validation completed


In [29]:
# Conditional Logic
def assign_age_group(age):
    """
    Categorize customers into age groups.
    """

    if pd.isna(age):
        return 'Unknown'

    elif age < 25:
        return '18-24'

    elif age < 35:
        return '25-34'

    elif age < 45:
        return '35-44'

    elif age < 55:
        return '45-54'

    else:
        return '55+'

In [30]:
# Applying the Function
customers['Age_Group'] = customers['Age'].apply(assign_age_group)

print("Customer age groups assigned successfully.")

Customer age groups assigned successfully.


In [31]:
# Check Result
print(
    customers[['customer_id', 'date_of_birth', 'Age', 'Age_Group']].head()
)


  customer_id date_of_birth   Age Age_Group
0  CUST002565    1957-07-03  69.0       55+
1  CUST004275    1966-12-07  59.0       55+
2  CUST003045    1998-06-14  28.0     25-34
3  CUST005168    1975-08-30  51.0     45-54
4  CUST003986    1999-09-15  26.0     25-34


In [32]:
# LAMBDA FUNCTION
median_price = products['unit_price'].median()

products['Price_Level'] = products['unit_price'].apply(
    lambda price: 'Low'
    if price < median_price
    else 'High'
)

In [33]:
# Checking Result
print(
    products[
        ['product_name', 'unit_price', 'Price_Level']
    ].head(10)
)

                      product_name  unit_price Price_Level
0         Cadence Wellness Classic      415.24        High
1          Northfield Cookware Pro      645.92        High
2  Pinecrest Accessories Essential         NaN        High
3          Vertex Camera Essential       70.18         Low
4              Lumen Wellness Plus      571.72        High
5            Orion Smartphone Plus       33.47         Low
6                  Crestline Decor      540.02        High
7       Meridian Basics Wellness X      119.71         Low
8            Lumen Accessories Pro      259.81         Low
9                   Nova Cameras X      304.03         Low


In [34]:
# CUSTOMER Order Analysis
## Order Count
orders_per_customer = (
    orders
    .groupby('customer_id')
    .size()
    .reset_index(name='Order_Count')
)

print(orders_per_customer.head())

  customer_id  Order_Count
0  CUST000001            6
1  CUST000002            4
2  CUST000003            4
3  CUST000004            9
4  CUST000005            6


In [35]:
# Customer Revenue
customer_revenue = (
    orders
    .groupby('customer_id')['total_amount']
    .sum()
    .reset_index(name='Total_Revenue')
)

print(customer_revenue.head())

  customer_id  Total_Revenue
0  CUST000001       17414.81
1  CUST000002        6922.38
2  CUST000003        4904.34
3  CUST000004        9441.76
4  CUST000005        8293.67


In [36]:
# Merge Customer Features
customers_analysis = customers.merge(
    orders_per_customer,
    on='customer_id',
    how='left'
)

customers_analysis = customers_analysis.merge(
    customer_revenue,
    on='customer_id',
    how='left'
)

In [37]:
# Handling customer without Order
customers_analysis['Order_Count'] = (
    customers_analysis['Order_Count']
    .fillna(0)
)

customers_analysis['Total_Revenue'] = (
    customers_analysis['Total_Revenue']
    .fillna(0)
)

In [38]:
# Customer segmentation
def classify_customer(order_count):
    """
    Classify customers according to order frequency.
    """

    if order_count == 0:
        return 'No Orders'

    elif order_count == 1:
        return 'One-Time Customer'

    elif order_count <= 5:
        return 'Repeat Customer'

    else:
        return 'High-Value Customer'

In [39]:
# Apply the classification
customers_analysis['Customer_Segment'] = (
    customers_analysis['Order_Count']
    .apply(classify_customer)
)

In [40]:
# Numpy Revenue Outlier Detection
mean_revenue = customers_analysis['Total_Revenue'].mean()

std_revenue = customers_analysis['Total_Revenue'].std()

customers_analysis['Revenue_ZScore'] = (
    customers_analysis['Total_Revenue']
    - mean_revenue
) / std_revenue

In [41]:
# Identify Outliers
customers_analysis['Revenue_Outlier'] = np.where(
    np.abs(customers_analysis['Revenue_ZScore']) > 3,
    'Yes',
    'No'
)


In [42]:
# Support Ticket Analysis
tickets_per_customer = (
    support_tickets
    .groupby('customer_id')
    .size()
    .reset_index(name='Ticket_Count')
)

In [43]:
# Aversge Satisfaction
ticket_satisfaction = (
    support_tickets
    .groupby('customer_id')['satisfaction_score']
    .mean()
    .reset_index(name='Average_Satisfaction')
)


In [44]:
# Merge Support Features
customers_analysis = customers_analysis.merge(
    tickets_per_customer,
    on='customer_id',
    how='left'
)

customers_analysis = customers_analysis.merge(
    ticket_satisfaction,
    on='customer_id',
    how='left'
)

In [45]:
# Handling Customers without Tickets
customers_analysis['Ticket_Count'] = (
    customers_analysis['Ticket_Count']
    .fillna(0)
)

In [46]:
# FINAL Data Quality Check
print("Final customer analysis shape:")
print(customers_analysis.shape)

print("\nFinal columns:")
print(customers_analysis.columns.tolist())

print("\nMissing values:")
print(customers_analysis.isnull().sum())

Final customer analysis shape:
(6090, 23)

Final columns:
['customer_id', 'first_name', 'last_name', 'email', 'phone', 'gender', 'date_of_birth', 'signup_date', 'state', 'city', 'membership_tier', 'preferred_channel', 'marketing_opt_in', 'referral_source', 'Age', 'Age_Group', 'Order_Count', 'Total_Revenue', 'Customer_Segment', 'Revenue_ZScore', 'Revenue_Outlier', 'Ticket_Count', 'Average_Satisfaction']

Missing values:
customer_id                0
first_name                 0
last_name                  0
email                    242
phone                    423
gender                     0
date_of_birth            181
signup_date              487
state                      0
city                     121
membership_tier            0
preferred_channel          0
marketing_opt_in        1191
referral_source         1705
Age                      354
Age_Group                  0
Order_Count                0
Total_Revenue              0
Customer_Segment           0
Revenue_ZScore            

In [47]:
# Preview the Final Customer Dataset
print(customers_analysis.head())

  customer_id first_name last_name                          email  \
0  CUST002565      Karen    Castro      karen.castro168@gmail.com   
1  CUST004275      Tonya   Webster     tonya.webster780@gmail.com   
2  CUST003045     Alexis    Thomas     alexis.thomas522@gmail.com   
3  CUST005168   Danielle   Goodman  danielle.goodman202@gmail.com   
4  CUST003986     Daniel    Bowman     daniel.bowman295@yahoo.com   

                 phone gender date_of_birth signup_date          state  \
0   784-499-9916x95359      F    1957-07-03  2026-05-28             FL   
1           3985078721      M    1966-12-07  2022-12-23  Massachusetts   
2           5624741570  Other    1998-06-14  2022-07-10      Wisconsin   
3   505.988.6915x03091      M    1975-08-30         NaT             in   
4  (988)305-0745x11524      F    1999-09-15  2023-07-05             TX   

          city  ...  referral_source   Age Age_Group Order_Count  \
0  Jeffreybury  ...   Email Campaign  69.0       55+         8.0   
1  M

In [48]:
# Original dataset
print(customers.head())

  customer_id first_name last_name                          email  \
0  CUST002565      Karen    Castro      karen.castro168@gmail.com   
1  CUST004275      Tonya   Webster     tonya.webster780@gmail.com   
2  CUST003045     Alexis    Thomas     alexis.thomas522@gmail.com   
3  CUST005168   Danielle   Goodman  danielle.goodman202@gmail.com   
4  CUST003986     Daniel    Bowman     daniel.bowman295@yahoo.com   

                 phone gender date_of_birth signup_date          state  \
0   784-499-9916x95359      F    1957-07-03  2026-05-28             FL   
1           3985078721      M    1966-12-07  2022-12-23  Massachusetts   
2           5624741570  Other    1998-06-14  2022-07-10      Wisconsin   
3   505.988.6915x03091      M    1975-08-30         NaT             in   
4  (988)305-0745x11524      F    1999-09-15  2023-07-05             TX   

          city membership_tier preferred_channel marketing_opt_in  \
0  Jeffreybury          SILVER            Online               no   
1 

In [49]:
# SAVE Processed Data
processed_datasets = {
    'customers_processed.csv': customers_analysis,
    'products_processed.csv': products,
    'orders_processed.csv': orders,
    'order_items_processed.csv': order_items,
    'support_tickets_processed.csv': support_tickets
}

In [50]:
# Save using a loop
for file_name, df in processed_datasets.items():

    file_path = os.path.join(
        processed_data_path,
        file_name
    )

    df.to_csv(
        file_path,
        index=False
    )

    print(f"Saved: {file_name}")

Saved: customers_processed.csv
Saved: products_processed.csv
Saved: orders_processed.csv
Saved: order_items_processed.csv
Saved: support_tickets_processed.csv


In [51]:
# Confirm saved files
print("Files in Processed Data:")

for file_name in os.listdir(processed_data_path):
    print(file_name)


Files in Processed Data:
customers_processed.csv
products_processed.csv
orders_processed.csv
order_items_processed.csv
support_tickets_processed.csv
